## Import Fungsi & Config

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

from src.utils.path_config import (
    ARTIFACTS_DIR,
)

from src.training.artifacts.load_training_metadata import (
    load_training_metadata
)

from src.visualization.loaders.load_attribution import (
    load_attribution
)

from src.preprocessing.config.preprocessing_config import (
    PreprocessingConfig
)

from src.training.config.evaluation_config import (
    EvaluationConfig
)

from src.xai.config.attribution_config import (
    AttributionConfig
)

from src.xai.pipeline.attribution_pipeline import (
    run_attribution
)

## Model Path

In [ ]:
MODEL_PATH = (
    ARTIFACTS_DIR /
    "models" /
    "model" /
    "training_baseline_artifacts" /
    "best_model.keras"
)

## Load Metadata

In [ ]:
metadata = load_training_metadata(

    MODEL_PATH.parent /
    "metadata.json"

)

## Set Configuration

### GradCAM Configuration

In [ ]:
attribution_config = AttributionConfig(

    experiment_name="model_gradcam_correct_only",

    xai_method="gradcam",

    correct_prediction_only=True,

    min_confidence=None,

)

### Preprocessing Configuration

In [ ]:
preprocessing_config = PreprocessingConfig(
  
    image_size=tuple(metadata["image_size"]),

    color_mode=metadata["color_mode"]

)

### Evaluation Configuration

In [ ]:
eval_config = EvaluationConfig(

    model_name=metadata["model_name"],

    preprocessing_config=preprocessing_config,

    batch_size=metadata["batch_size"],

    seed=metadata["seed"],

    num_classes=metadata["num_classes"],

    test_csv=Path(metadata["test_csv"])

)

## Run Attribution

In [ ]:
run_attribution(

    model_path=MODEL_PATH,

    metadata_path=
        MODEL_PATH.parent /
        "metadata.json",

    evaluation_config=eval_config,

    attribution_config=attribution_config,

    lime_config=None

)

## Attribution Information Schema

In [ ]:
npz_path = Path(
    ARTIFACTS_DIR /
    "attributions" /
    "model_gradcam_correct_only_20260614_153522" /
    "attribution_maps" /
    "batik_betawi_3b786187748de877.npz"
)

data = load_attribution(npz_path)

print(data.keys())

rows = []

for key, value in data.items():

    rows.append({
        "field": key,
        "type": type(value).__name__,
        "shape": (
            value.shape
            if isinstance(value, np.ndarray)
            else None
        ),
        "dtype": (
            str(value.dtype)
            if isinstance(value, np.ndarray)
            else None
        )
    })

pd.DataFrame(rows)